# Word EEG: classification and similarity in one run
**Run this after `prepare_trials.ipynb` completes all 48 recordings. CPU → Runtime → Run all.**

This reads the prepared 40-Hz /125-Hz trial archive, not source EEG. Main comparison: **positive vs negative**. Secondary: **positive / neutral / negative**.

- **Input:** six scalp regions × eight 100-ms windows = 48 EEG values per trial. The full archive stays unchanged for other feature choices.
- **Models:** balanced logistic regression, shrinkage LDA, and a baseline that ignores EEG. Fixed settings, no tuning sweep.
- **Testing:** hold out sentence texts, word identities, or readers separately. Word splits also purge overlapping training EEG windows. Inadequate splits are explicitly skipped.
- **Controls:** pre-onset EEG, word/fixation properties, six individual regions, and the four content-word label policies we discussed.
- **Similarity:** same-class/different-class correlations; repeated-word comparisons within readers; small trial matrices; reader reproducibility.

The main population uses word-alone = word-in-context labels, with no whole-sentence requirement, and keeps neutral/function words and control reading. Auxiliary policies require content-word labels: all-three agreement, lexical only, contextual-word only, sentence only. Missing private labels/grammar or sparse classes cause a reported skip. No new labels are guessed.

Output: `Thesis/Results/word_sentiment_eeg/classification_similarity/`. Expect roughly **100–250 MB additional storage**, depending on trial counts; a 400-MB check stops an unexpectedly large run. No GPU, source downloads or re-extraction. First-run archive reading takes time; validated compact features and completed model cases are reused on rerun.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys, json, html
import pandas as pd
from IPython.display import display, Markdown, Image, HTML

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
ARCHIVE=THESIS/'Results/word_sentiment_eeg/trial_archive_125hz'
OUT=THESIS/'Results/word_sentiment_eeg/classification_similarity'
state=json.loads((ARCHIVE/'manifest.json').read_text())
if state.get('status')!='completed' or state.get('completed_reader_recordings')!=48:
    raise RuntimeError('Preparation is not fully complete. Wait for all 48 recordings; no analysis has started.')
REPO=Path('/content/word-sentiment-eeg-classification')
def git(*args):
    subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],check=True,
                   env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary analysis checkout has edits; preserve them before updating.')
    git('-C',REPO,'pull','--ff-only','origin','main')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-analysis.txt')],check=True)
print('Analysis code:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'],text=True).strip())
print('Archive complete:',state['completed_reader_recordings'],'recordings')


## Run the fixed analysis battery
All datasets/tasks run together, but their results stay separate. No neural-network training or automatic parameter search. A model fit or a skipped experiment is logged; quiet periods show a heartbeat.


In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/analyze_trial_archive.py'),'--thesis-root',str(THESIS)]
run_with_progress(command,OUT/'runtime_status.json',activity='Fixed EEG classification and similarity analyses')


## Completion, methods and aggregate input counts
Balanced accuracy gives equal importance to the classes: reference chance is **0.50 binary**, **0.33 three-class**. These are exploratory out-of-fold results, not proof of sentiment-specific brain coding. Each held-out scheme answers a different question.


In [ ]:
result=json.loads((OUT/'manifest.json').read_text())
if result.get('status')!='completed':
    raise RuntimeError('Analysis is incomplete. Saved features and completed model cases can resume.')
print('Completed dataset/tasks:',result['tasks_complete'])
COUNTS=pd.read_csv(OUT/'input_counts.csv')
display(COUNTS[(COUNTS.subject=='ALL_READERS')&(COUNTS.policy=='word_agreement')&(COUNTS.target=='three_class')])
display(HTML('<details><summary><b>Methods, label policies, split definitions and limitations</b></summary><pre style="white-space:pre-wrap">'+html.escape((OUT/'REPORT.md').read_text())+'</pre></details>'))

def read_table(path):
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

def show_image(folder,name):
    path=folder/'plots'/(name+'.png')
    if path.is_file():display(Image(filename=str(path)))
    else:print('Not available:',name,'— see skipped.json and coverage counts.')

def show_task(dataset,task,section):
    folder=OUT/dataset/task
    scores=read_table(folder/'scores.csv')
    if section=='classification':
        if len(scores):
            selected=scores[(scores.policy=='word_agreement')&(scores.feature=='post')]
            display(selected[['target','split','model','trials','readers','balanced_accuracy','equal_reader_balanced_accuracy','macro_f1','accuracy']])
        for name in ['classification_overview','binary_confusions','three_class_confusions']:show_image(folder,name)
    elif section=='controls':
        display(COUNTS[(COUNTS.dataset==dataset)&(COUNTS.task==task)&(COUNTS.subject=='ALL_READERS')])
        for name in ['regions_and_controls','label_policy_comparison']:show_image(folder,name)
        skipped=json.loads((folder/'skipped.json').read_text())
        if skipped:
            print('Experiments skipped because of class/group coverage:')
            display(pd.DataFrame(skipped))
        quality=read_table(OUT/'quality_counts.csv')
        display(quality[(quality.dataset==dataset)&(quality.task==task)])
    elif section=='similarity':
        for name in ['similarity_overview','sampled_trial_matrix','reader_reproducibility']:show_image(folder,name)
    elif section=='readers':
        show_image(folder,'reader_score_matrix')
        print('Seen-word diagnostic: test word also appeared for the SAME reader in training, in a different sentence. Pooled model; distinct EEG events.')
        display(read_table(folder/'same_word_seen_diagnostic.csv'))
        print('Repeated-word correlation: one row per reader, not one independent subject per pair.')
        path=folder/'repeated_word_reader_means.csv'
        if path.exists():display(read_table(path))
        print('All per-reader class counts:')
        display(COUNTS[(COUNTS.dataset==dataset)&(COUNTS.task==task)&(COUNTS.subject!='ALL_READERS')&(COUNTS.policy=='word_agreement')&(COUNTS.target=='three_class')].pivot(index='subject',columns='class_label',values='trials'))


## ZUCO SR — Binary and three-class classification


In [ ]:
show_task('zuco','SR','classification')


## ZUCO SR — Regions, label policies and confound controls


In [ ]:
show_task('zuco','SR','controls')


## ZUCO SR — Class / repeated-word / reader similarity


In [ ]:
show_task('zuco','SR','similarity')


## ZUCO SR — Reader-specific scores and repeated-word diagnostics


In [ ]:
show_task('zuco','SR','readers')


## ZUCO NR — Binary and three-class classification


In [ ]:
show_task('zuco','NR','classification')


## ZUCO NR — Regions, label policies and confound controls


In [ ]:
show_task('zuco','NR','controls')


## ZUCO NR — Class / repeated-word / reader similarity


In [ ]:
show_task('zuco','NR','similarity')


## ZUCO NR — Reader-specific scores and repeated-word diagnostics


In [ ]:
show_task('zuco','NR','readers')


## ZUCO TSR — Binary and three-class classification


In [ ]:
show_task('zuco','TSR','classification')


## ZUCO TSR — Regions, label policies and confound controls


In [ ]:
show_task('zuco','TSR','controls')


## ZUCO TSR — Class / repeated-word / reader similarity


In [ ]:
show_task('zuco','TSR','similarity')


## ZUCO TSR — Reader-specific scores and repeated-word diagnostics


In [ ]:
show_task('zuco','TSR','readers')


## TECO task1 — Binary and three-class classification


In [ ]:
show_task('teco','task1','classification')


## TECO task1 — Regions, label policies and confound controls


In [ ]:
show_task('teco','task1','controls')


## TECO task1 — Class / repeated-word / reader similarity


In [ ]:
show_task('teco','task1','similarity')


## TECO task1 — Reader-specific scores and repeated-word diagnostics


In [ ]:
show_task('teco','task1','readers')


## Saved files and interpretation
- `scores.csv`: all fixed comparisons. `input_counts.csv`: class counts summed across readers, unique source occurrences, and per-reader counts.
- `quality_counts.csv` and task `excluded_duplicate_signals.csv.gz`: explicit quality/duplicate accounting. No neutral downsampling.
- Per-task `cases/`: held-out predictions, fold IDs, metrics and confusion counts; completed fits are reused.
- Per-task pair CSVs and plots: bounded descriptive correlations. Many pairs from one reader are **not** many independent readers.
- `features/`: small reusable features. The original all-channel trial archive is read-only.

All-three agreement can leave too little neutral data. Blank/skipped comparisons are coverage limitations, not zero accuracy. Pre-onset or nuisance performance suggests possible confounding and needs interpretation. Similarity may reflect common reading responses, lexical/context effects or residual artifacts, not sentiment alone. Region comparisons are exploratory; do not select a winning region/model and call the same held-out score a new confirmatory result.


In [ ]:
print('Results folder:',OUT)
print('All scores:',OUT/'scores.csv')
print('Counts:',OUT/'input_counts.csv')
print('Skipped comparisons:',OUT/'skipped.json')
